# Basic RNN - Introduction & Time Series

We will build a **Basic RNN Predictor** from scratch using PyTorch. 

We will use a synthetic dataset (Simulated Flight Passengers or Sales Data) to ensure everyone has the same clean data to learn from.


### 1. Import Libraries
As always, we need our tools. Notice we are importing `torch.nn`.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

torch.manual_seed(42)
np.random.seed(42)

### 2. Generating Data
Instead of downloading a complex CSV file, let's generate a clean **Sales Trend** with some noise using Pandas. This represents a perfect seasonal pattern (like hourly temperature or yearly sales).

We will generate 120 data points (simulating 10 monthly years).

In [ ]:
# 1. Create a Date Range
dates = pd.date_range(start='2024-01-01', periods=120, freq='ME') # ME: Monthly End

# 2. Create values with Trend + Seasonality
x = np.linspace(0, 10, 120)

# Trend: linear growth (x * 2)
# Seasonality: Sine wave (np.sin(x * 3))
# Noise: Random
values = (x * 2) + (np.sin(x * 5) * 5) + np.random.normal(0, 1.5, 120)  

# 3. Create DataFrame
df = pd.DataFrame({'Date': dates, 'Value': values})

# Display first few rows to simulate "Real Data"
print(df.head())
print(df.describe())

# Extract values for the model
y = df['Value'].values.astype(float)

plt.figure(figsize=(12, 4))
plt.title("Synthetic Monthly Sales Data (Trend + Seasonality)")
plt.plot(df['Date'], y)
plt.ylabel("Sales")
plt.xlabel("Date")
plt.grid(True)
plt.show()

### 3. Data Preprocessing

Neural Networks work best when data is small, typically between -1 and 1 or 0 and 1. We will use `MinMaxScaler`.

In [ ]:
# Normalize data to be between -1 and 1

scaler = MinMaxScaler(feature_range=(-1, 1))
y_normalized = scaler.fit_transform(y.reshape(-1, 1))

# Convert to PyTorch Tensor
y_tensor = torch.FloatTensor(y_normalized).view(-1) # view(-1) makes it a 1D tensor

print(f"Original shape: {y.shape}")
print(f"Tensor shape: {y_tensor.shape}")

#### Creating Sequences (The Sliding Window)
We need to organize our data into `(Input, Label)` pairs.

If our window size is 10:
- **Input:** Days 1 to 10
- **Label (Target):** Day 11

Next pair:
- **Input:** Days 2 to 11
- **Label (Target):** Day 12

This creates a dataset where the model learns: "Given these 10 days, what happens next?"

In [ ]:
def create_sequences(input_data, window_size):
    sequences = []
    labels = []
    
    L = len(input_data) 
    for i in range(L - window_size):
        # Retrieve the sequence of length 'window_size'
        seq = input_data[i:i+window_size]

        # The target is the very next value after the sequence
        label = input_data[i+window_size]
        
        sequences.append(seq)
        labels.append(label)
        
    return torch.stack(sequences), torch.stack(labels)

# Settings
window_size = 12  # Look back at the last 12 points (1 year) to predict the next one

X, z = create_sequences(y_tensor, window_size)

print(f"Input shape (X): {X.shape}  <- (Total Sequences, Window Size)")
print(f"Target shape (z): {z.shape}  <- (Total Targets)")

### 4. Train/Test Split
We can't shuffle time series data! (Why? Because the future cannot predict the past in training). We must split sequentially.

In [ ]:
test_size = 24
train_size = len(X) - test_size

train_X = X[:train_size]
train_y = z[:train_size]

test_X = X[train_size:]
test_y = z[train_size:]

print(f"Training sets: {train_X.shape}")
print(f"Testing sets: {test_X.shape}")

### 5. Defining the Model: Vanilla RNN

We will use `nn.RNN`, the most basic Recurrent Neural Network provided by PyTorch.

**Architecture:**
1. **RNN Layer**: Reads the sequence step-by-step.
2. **Linear Layer**: Takes the final hidden state of the RNN and maps it to a single value (our prediction).

In [ ]:
class TimeSeriesRNN(nn.Module):
    def __init__(self, input_size=1, hidden_size=50, output_size=1):
        super().__init__()
        self.hidden_size = hidden_size
        
        # input_size = 1 (we are only passing 1 value per time step: the sine value)
        # hidden_size = 50 (the number of features the RNN 'learns' to represent the pattern)
        # For current data, hidden_size is the number of features the RNN 'learns' to represent the pattern 
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        
        # The Linear Layer (Fully Connected)
        # Maps the 50 hidden features to 1 output value
        self.linear = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_features)
        # We need to reshape x to have the input_features dimension, usually it is [Batch, Sequence] -> [Batch, Sequence, 1]
        x = x.view(len(x), -1, 1)
        
        # RNN returns: out, hidden_state
        # out shape: (batch_size, seq_len, hidden_size)
        rnn_out, _ = self.rnn(x)
        
        # We only want the output of the LAST time step in the sequence
        # (Specifically, we want to know what comes AFTER the sequence ends)
        last_time_step = rnn_out[:, -1, :]
        
        # Pass through linear layer to get prediction
        predictions = self.linear(last_time_step)
        
        return predictions

### In the `forward` function, why did we do `rnn_out[:, -1, :]`?

The RNN gives us an output for *every* step in the sequence (Day 1, Day 2... Day 40). 

But we only care about the final accumulated "memory" after seeing all 40 days to predict Day 41. So, we take the last one!

In [ ]:
# Instantiate the model
model = TimeSeriesRNN()

# Loss Function: MSE (Mean Squared Error) is standard for Regression numbers
criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(model)

### 6. Training Loop with 100 epochs

In [ ]:
epochs = 60
train_losses = []
test_losses = []

print("Starting Training...")

for i in range(epochs):
    
    # --- Training Phase ---
    model.train()
    
    optimizer.zero_grad()
    
    y_pred = model(train_X)
    loss = criterion(y_pred.view(-1), train_y) # flatten prediction to match target
    
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    # Evaluation Phase (Optional inside loop, but good for checking overfit)
    model.eval()
    with torch.no_grad():
        test_pred = model(test_X)
        test_loss = criterion(test_pred.view(-1), test_y)
        test_losses.append(test_loss.item())

    if i % 10 == 0:
        print(f'Epoch {i:3} | Train Loss: {loss.item():.5f} | Test Loss: {test_loss.item():.5f}')

In [ ]:
# Plotting Loss

plt.figure(figsize=(10,5))
plt.plot(train_losses, label='Training Loss')
plt.plot(test_losses, label='Test Loss')
plt.title("Loss over Epochs")
plt.legend()
plt.show()

### 7. Evaluation & Forecasting

Now, let's see how well our model predicts the future. 

We will visualize the predictions on the Test Set (which the model never saw during training).

In [ ]:
model.eval()
with torch.no_grad():
    # Predict on the test data
    predicted_norm = model(test_X).numpy()

# Inverse transform to get back to original scale (un-normalize)
predicted_real = scaler.inverse_transform(predicted_norm)
actual_real = scaler.inverse_transform(test_y.reshape(-1, 1))

plt.figure(figsize=(12, 5))
plt.title("Prediction vs Reality")
plt.plot(actual_real, label="Actual Data")
plt.plot(predicted_real, label="RNN Prediction")
plt.legend()
plt.grid(True)
plt.show()

### TODO

Compare this Basic RNN with LSTM. 

Does the Basic RNN struggle finding patterns if you make the noise higher in the dataset? Or if the sequence is much longer (e.g. 100)?

### Conclusion

1. Generated sequential data (Sine wave).
2. Created "Windows" of data (Look back 40 days and predict Day 41).
3. Feed the sequence into an `nn.RNN` layer.
4. Used the final hidden state to predict the value.